# mintkit data sources: what they are and how they work

This notebook walks through `mintkit/sources/`, the layer that feeds documents, queries and relevance
judgments into the minting pipeline. It needs no model and no network: every example runs on local MLDR files
or on small temporary fixtures that the notebook builds itself.

Contents:
1. The contract: `Doc`, `Query`, `Qrels`, `Source`
2. The registry and spec strings (`kind:value`)
3. `folder:` sources
4. `mldr:` sources (corpus mode vs positives mode)
5. `beir:` sources
6. `msmarco:` sources (optional `ir_datasets`)
7. Sampling: `all`, `first:N`, `random:N`, `qrels:Q+D`, `ids:FILE`
8. From a `Doc` to the text that gets embedded (`prepare_card`)
9. Writing your own source

In [1]:
import os, sys, json, tempfile, textwrap
from pathlib import Path
from itertools import islice

# Run from the repository root so relative paths such as data/mldr resolve.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mintkit.sources import SOURCES, load_source, select, Doc, Query, Source
from mintkit.sources.base import Selection, reservoir, DEFAULT_SEED

def show_doc(doc, width=100):
    body = doc.text if doc.text is not None else f"<file {doc.path}>"
    print(f"doc_id={doc.doc_id!r} title={doc.title!r} meta={doc.meta}")
    print("   ", textwrap.shorten(body, width))

TMP = Path(tempfile.mkdtemp(prefix="mintkit-sources-"))
print("repo:", ROOT)
print("scratch fixtures:", TMP)

repo: /home/vincesz/clawmarc
scratch fixtures: /tmp/mintkit-sources-55kcak0_


## 1. The contract

Every source produces the same three things:

| Type | Shape | Notes |
|---|---|---|
| `Doc` | `doc_id`, and **exactly one** of `text` (dataset docs) or `path` (files), plus optional `title`, `meta` | frozen dataclass; checks its invariants in `__post_init__` |
| `Query` | `qid`, `text` | |
| `Qrels` | `dict[qid, dict[doc_id, grade]]` | grade > 0 counts as relevant |

A `Source` must implement `iter_docs()` (a **stream**; it must never load the corpus into RAM) and `describe()`
(which goes into the run manifest). `splits()`, `queries(split)`, `qrels(split)`, `get_docs(ids)` and `count()`
have defaults: no splits, no judgments, and `get_docs` falls back to a linear scan.

In [2]:
print(Doc("a", text="hello"))
print(Doc("b", path=Path("x.pdf")))
for bad in (dict(doc_id="c"), dict(doc_id="d", text="t", path=Path("p")), dict(doc_id="", text="t")):
    try:
        Doc(**bad)
    except ValueError as exc:
        print(f"Doc({bad}) -> ValueError: {exc}")

Doc(doc_id='a', text='hello', path=None, title=None, meta={})
Doc(doc_id='b', text=None, path=PosixPath('x.pdf'), title=None, meta={})
Doc({'doc_id': 'c'}) -> ValueError: A document must have exactly one of text or path
Doc({'doc_id': 'd', 'text': 't', 'path': PosixPath('p')}) -> ValueError: A document must have exactly one of text or path
Doc({'doc_id': '', 'text': 't'}) -> ValueError: A document must have an ID


## 2. Registry and spec strings

Sources are addressed by `kind:value`. `load_source(spec, **options)` splits on the first `:` and calls the
registered class with `value` plus keyword options. The CLI passes extra options (`--mode`, `--glob`, ...) the same way.
`beir:NAME` resolves to `data/beir/NAME` when `NAME` is not an existing path.

In [3]:
for kind, cls in SOURCES.items():
    print(f"{kind:8s} -> {cls.__module__}.{cls.__name__}")

for spec in ("mldr", "nope:x", "mldr:"):
    try:
        load_source(spec)
    except ValueError as exc:
        print(f"{spec!r}: {exc}")

folder   -> mintkit.sources.folder.FolderSource
mldr     -> mintkit.sources.mldr.MLDRSource
beir     -> mintkit.sources.beir.BEIRSource
msmarco  -> mintkit.sources.msmarco.MSMarcoSource
'mldr': Invalid source 'mldr'; expected kind:value with kind in folder, mldr, beir, msmarco
'nope:x': Invalid source 'nope:x'; expected kind:value with kind in folder, mldr, beir, msmarco
'mldr:': Invalid source 'mldr:'; expected kind:value with kind in folder, mldr, beir, msmarco


## 3. `folder:PATH`

* Recursive glob (`--glob '*.pdf'`; the default `*` becomes `**/*`), sorted, so the order is stable.
* `doc_id` is the POSIX path relative to the root. The `Doc` carries a `path`; bytes are read later, in a prepare worker.
* Hidden files and directories are skipped unless `include_hidden=True`.
* Directories that contain a mintkit run (`manifest.json` with `schema: mintkit-run-1`) are skipped, so minting into a
  `runs/` folder inside the source does not mint its own output.
* Optional `queries.tsv` (qid, text) and `qrels.tsv` (qid, doc_id, grade; 4-column TREC format also accepted) in the
  root give the source one split called `default`.

In [4]:
folder = TMP / "library"
(folder / "papers").mkdir(parents=True)
(folder / ".git").mkdir()
(folder / "old_run").mkdir()
(folder / "papers" / "a.txt").write_text("Cats are small domesticated carnivores.", encoding="utf-8")
(folder / "papers" / "b.txt").write_text("Rust is a systems programming language.", encoding="utf-8")
(folder / "notes.md").write_text("# Notes\n\nThe cat sat on the mat.", encoding="utf-8")
(folder / ".git" / "HEAD").write_text("ref: refs/heads/main", encoding="utf-8")
(folder / "old_run" / "manifest.json").write_text(json.dumps({"schema": "mintkit-run-1"}), encoding="utf-8")
(folder / "old_run" / "card.txt").write_text("generated output, must be skipped", encoding="utf-8")
(folder / "queries.tsv").write_text("qid\ttext\nq1\twhat is a cat\nq2\tsystems language\n", encoding="utf-8")
(folder / "qrels.tsv").write_text("q1\tpapers/a.txt\t1\nq1\tnotes.md\t1\nq2\tpapers/b.txt\t1\n", encoding="utf-8")

src = load_source(f"folder:{folder}")
print(src.name, "splits:", src.splits())
for doc in src.iter_docs():
    show_doc(doc)
print("queries:", src.queries("default"))
print("qrels:", src.qrels("default"))
print("only *.txt:", [d.doc_id for d in load_source(f"folder:{folder}", glob="*.txt").iter_docs()])
print("describe():", json.dumps(src.describe(), indent=1)[:400])

folder:library splits: ['default']
doc_id='notes.md' title=None meta={}
    <file /tmp/mintkit-sources-55kcak0_/library/notes.md>
doc_id='papers/a.txt' title=None meta={}
    <file /tmp/mintkit-sources-55kcak0_/library/papers/a.txt>
doc_id='papers/b.txt' title=None meta={}
    <file /tmp/mintkit-sources-55kcak0_/library/papers/b.txt>
queries: [Query(qid='q1', text='what is a cat'), Query(qid='q2', text='systems language')]
qrels: {'q1': {'papers/a.txt': 1, 'notes.md': 1}, 'q2': {'papers/b.txt': 1}}
only *.txt: ['papers/a.txt', 'papers/b.txt']
describe(): {
 "root": "/tmp/mintkit-sources-55kcak0_/library",
 "glob": "*",
 "include_hidden": false,
 "files": {
  "/tmp/mintkit-sources-55kcak0_/library/queries.tsv": "2ae4a1e461354ac944e96706a77b5ae3c23ccf480350ab302339ce133c6e09d4",
  "/tmp/mintkit-sources-55kcak0_/library/qrels.tsv": "95557452a1725f37912b71557d60ad403ef9769b3bb63cf459acb1388be0c8c4"
 }
}


## 4. `mldr:LANG`

MLDR (`Shitao/MLDR` on Hugging Face) ships per language: `corpus.jsonl.gz` (about 200k long Wikipedia docs for `en`,
roughly 1 GB) and `train/dev/test.jsonl.gz`, where each row is a query with its `positive_passages`
(and negatives). The loader reads `data/mldr/mldr-v1.0-LANG/` first and only calls `hf_hub_download` for files that are missing.

Two modes:

| mode | documents | needs corpus file? |
|---|---|---|
| `corpus` (default) | every row of `corpus.jsonl.gz` | yes |
| `positives` | the de-duplicated positive passages of **one split** | no, only the split file |

In `positives` mode the documents are bound to the split (`document_split`), so `select()` refuses a different query split.
MLDR has no titles, so the pipeline gives these docs the synthetic title `"mldr:en doc-en-8"`, which is shown on the card but not embedded.

In [5]:
mldr_dir = ROOT / "data" / "mldr"
print("local MLDR languages:", sorted(p.name for p in mldr_dir.glob("mldr-v1.0-*")) if mldr_dir.is_dir() else "none")
print("en files:", sorted(p.name for p in (mldr_dir / "mldr-v1.0-en").glob("*")) if (mldr_dir / "mldr-v1.0-en").is_dir() else "none")

mldr = load_source("mldr:en", mode="positives", split="dev")
print(mldr.name, "mode:", mldr.mode, "document_split:", mldr.document_split, "splits:", mldr.splits())

local MLDR languages: ['mldr-v1.0-de', 'mldr-v1.0-en', 'mldr-v1.0-es', 'mldr-v1.0-fr', 'mldr-v1.0-ru', 'mldr-v1.0-zh']
en files: ['dev.jsonl.gz', 'test.jsonl.gz']
mldr:en mode: positives document_split: dev splits: ['train', 'dev', 'test']


In [6]:
docs = list(islice(mldr.iter_docs(), 3))
for doc in docs:
    show_doc(doc)

queries, qrels = mldr.queries("dev"), mldr.qrels("dev")
print(f"\ndev: {len(queries)} queries, {sum(len(v) for v in qrels.values())} judgments")
q = queries[0]
print(q, "->", qrels[q.qid])

doc_id='doc-en-8' title=None meta={}
    In computer programming, is a ternary operator that is part of the syntax for basic [...]
doc_id='doc-en-391' title=None meta={}
    KPMG International Limited (or simply KPMG) is a British-Dutch multinational professional [...]
doc_id='doc-en-431' title=None meta={}
    The kidnapping of Alan Johnston, a British journalist for the BBC, by the Palestinian Army of [...]



dev: 200 queries, 200 judgments
Query(qid='q-en-1', text='What is the syntax for the shorthand of the conditional operator in PHP 5.3?') -> {'doc-en-8': 1}


In [7]:
# How long are MLDR documents? This is why minting is embedding-bound: about 9k tokens, roughly 20 chunks of 480 tokens per doc.
import statistics
lengths = [len(d.text) for d in islice(mldr.iter_docs(), 200)]
print(f"{len(lengths)} docs, chars: median {statistics.median(lengths):,.0f}, min {min(lengths):,}, max {max(lengths):,}")

200 docs, chars: median 29,868, min 16,761, max 100,475


In [8]:
# Positives mode is tied to the split it was opened with:
try:
    select(mldr, "all", split="test")
except ValueError as exc:
    print("ValueError:", exc)

# Corpus mode: only touches corpus.jsonl.gz when you iterate. Here it is only constructed, so nothing is downloaded.
corpus = load_source("mldr:en")
local_corpus = (mldr_dir / "mldr-v1.0-en" / "corpus.jsonl.gz").is_file()
print("corpus mode constructed; local corpus file present:", local_corpus,
      "(iterating would download it)" if not local_corpus else "")

ValueError: mldr:en documents are from split 'dev', not 'test'
corpus mode constructed; local corpus file present: False (iterating would download it)


## 5. `beir:PATH_OR_NAME`

The standard BEIR layout: `corpus.jsonl` (`_id`, `title`, `text`, optional `metadata`), `queries.jsonl` shared by all
splits, and `qrels/<split>.tsv` (header `query-id corpus-id score`). Splits are the file stems in `qrels/`;
`queries(split)` returns only the queries that have judgments in that split. This covers SciFact, NFCorpus, TREC-COVID
and others without extra code. Below is a tiny synthetic BEIR set.

In [9]:
beir = TMP / "tinybeir"
(beir / "qrels").mkdir(parents=True)
corpus_rows = [
    {"_id": "d1", "title": "Cats", "text": "Cats purr when content."},
    {"_id": "d2", "title": "Dogs", "text": "Dogs bark at strangers."},
    {"_id": "d3", "title": "Rust", "text": "Rust has a borrow checker."},
    {"_id": "d4", "title": "Go", "text": "Go has goroutines.", "metadata": {"year": 2009}},
]
(beir / "corpus.jsonl").write_text("".join(json.dumps(r) + "\n" for r in corpus_rows), encoding="utf-8")
(beir / "queries.jsonl").write_text(
    "".join(json.dumps(r) + "\n" for r in [{"_id": "q1", "text": "why do cats purr"}, {"_id": "q2", "text": "memory safety"}]),
    encoding="utf-8")
(beir / "qrels" / "test.tsv").write_text("query-id\tcorpus-id\tscore\nq1\td1\t1\nq2\td3\t2\nq2\td4\t0\n", encoding="utf-8")
(beir / "qrels" / "dev.tsv").write_text("query-id\tcorpus-id\tscore\nq1\td1\t1\n", encoding="utf-8")

b = load_source(f"beir:{beir}")
print(b.name, "splits:", b.splits())
for doc in b.iter_docs():
    show_doc(doc)
print("test queries:", b.queries("test"))
print("dev queries:", b.queries("dev"))
print("test qrels:", b.qrels("test"), "<- grade 0 is kept but is not relevant")

beir:tinybeir splits: ['dev', 'test']
doc_id='d1' title='Cats' meta={}
    Cats purr when content.
doc_id='d2' title='Dogs' meta={}
    Dogs bark at strangers.
doc_id='d3' title='Rust' meta={}
    Rust has a borrow checker.
doc_id='d4' title='Go' meta={'year': 2009}
    Go has goroutines.
test queries: [Query(qid='q1', text='why do cats purr'), Query(qid='q2', text='memory safety')]
dev queries: [Query(qid='q1', text='why do cats purr')]
test qrels: {'q1': {'d1': 1}, 'q2': {'d3': 2, 'd4': 0}} <- grade 0 is kept but is not relevant


## 6. `msmarco:passage`

A thin adapter over `ir_datasets` (optional: `pip install -e .[msmarco]`). 8.8M short passages (about 60 tokens, so one chunk each);
splits `train`, `dev/small`, `trec-dl-2019`, `trec-dl-2020`. `get_docs(ids)` uses the `ir_datasets` docstore for
random access instead of a scan, which `qrels:Q` sampling without distractors benefits from. Passage docs have no title.

In [10]:
try:
    import ir_datasets  # noqa: F401
    ms = load_source("msmarco:passage")
    print(ms.name, ms.splits(), ms.describe())
    print("(not iterating: the first iteration downloads the collection, about 1 GB)")
except Exception as exc:
    print("ir_datasets is not installed here:", exc)
    print("The adapter still constructs:", load_source("msmarco:passage").name, load_source("msmarco:passage").splits())

ir_datasets is not installed here: No module named 'ir_datasets'
The adapter still constructs: msmarco:passage ['train', 'dev/small', 'trec-dl-2019', 'trec-dl-2020']


## 7. Sampling

`select(source, sample, split=..., seed=...)` returns a `Selection`: a **re-iterable** `iter_docs` callable plus the split's
`queries` and `qrels`. The pipeline calls `iter_docs()` once per model, so a sample must give the same documents every time
(all samplers here are deterministic given the seed; the default seed is 20260910).

| spec | what you get | memory |
|---|---|---|
| `all` | the whole stream | O(1) |
| `first:N` | the first N in source order. Not random: MLDR's first lines are the dev/test docs | O(1) |
| `random:N` | uniform reservoir sample, **returned in source order** | O(N) docs |
| `qrels:Q[+D]` | Q random queries with at least one positive, all their positives, plus D random non-positive distractors | O(Q+D) |
| `ids:FILE` | exactly these doc_ids (one per line); errors if any are missing | O(ids) |

After minting, the run writes `queries.jsonl` and `qrels.tsv` restricted to documents that were actually minted
(`Selection.restrict_qrels`), dropping queries that no longer have a relevant doc.

In [11]:
sel = select(mldr, "first:5", split="dev")
print("first:5  ", [d.doc_id for d in sel.iter_docs()])

sel = select(mldr, "random:5", split="dev")
a = [d.doc_id for d in sel.iter_docs()]
b_ = [d.doc_id for d in sel.iter_docs()]
print("random:5 ", a, "| re-iterable & deterministic:", a == b_)
print("random:5 seed=1", [d.doc_id for d in select(mldr, "random:5", split="dev", seed=1).iter_docs()])

first:5   ['doc-en-8', 'doc-en-391', 'doc-en-431', 'doc-en-637', 'doc-en-652']


random:5  ['doc-en-865', 'doc-en-8687', 'doc-en-10633', 'doc-en-15861', 'doc-en-16191'] | re-iterable & deterministic: True


random:5 seed=1 ['doc-en-431', 'doc-en-9224', 'doc-en-12357', 'doc-en-103', 'doc-en-13961']


In [12]:
# The MLDR test-plan collection design: Q queries, their positives, and D distractors.
sel = select(mldr, "qrels:5+20", split="dev")
ids = [d.doc_id for d in sel.iter_docs()]
relevant = {doc for j in sel.qrels.values() for doc, g in j.items() if g > 0}
print(f"{len(sel.queries)} queries, {len(ids)} docs: {len(relevant)} positives first, then {len(ids) - len(relevant)} distractors")
print("positives at the front:", set(ids[:len(relevant)]) == relevant)

# What a run keeps if, say, only the first 22 docs got minted:
kept = sel.restrict_qrels(set(ids[:22]))
print("restricted qrels:", kept)

5 queries, 25 docs: 5 positives first, then 20 distractors
positives at the front: True
restricted qrels: {'q-en-284': {'doc-en-2222': 1}, 'q-en-419': {'doc-en-3266': 1}, 'q-en-483': {'doc-en-3747': 1}, 'q-en-729': {'doc-en-5637': 1}, 'q-en-2188': {'doc-en-14979': 1}}


In [13]:
# ids:FILE, and the error you get for an unknown id
idfile = TMP / "ids.txt"
idfile.write_text("doc-en-8\ndoc-en-391\n", encoding="utf-8")
print("ids:FILE ->", [d.doc_id for d in select(mldr, f"ids:{idfile}", split="dev").iter_docs()])
idfile.write_text("doc-en-8\ndoes-not-exist\n", encoding="utf-8")
try:
    list(select(mldr, f"ids:{idfile}", split="dev").iter_docs())
except ValueError as exc:
    print("ValueError:", exc)

# qrels sampling needs a split; asking for more queries than exist is an error
for spec, split in (("qrels:3", None), ("qrels:99999", "dev")):
    try:
        select(mldr, spec, split=split)
    except ValueError as exc:
        print(f"{spec} split={split}: {exc}")

ids:FILE -> ['doc-en-8', 'doc-en-391']


ValueError: mldr:en is missing 1 requested document IDs: does-not-exist
qrels:3 split=None: qrels sampling requires a query split
qrels:99999 split=dev: mldr:en has only 200 queries with positive judgments


In [14]:
# The reservoir itself: uniform over the stream, O(size) memory, output in stream order.
from random import Random
from collections import Counter
hits = Counter()
for trial in range(2000):
    hits.update(d.doc_id for d in reservoir((Doc(str(i), text="x") for i in range(10)), 3, Random(trial)))
print("each of 10 items should be picked about 600 times:", dict(sorted(hits.items(), key=lambda kv: int(kv[0]))))

each of 10 items should be picked about 600 times: {'0': 586, '1': 612, '2': 601, '3': 626, '4': 627, '5': 572, '6': 609, '7': 593, '8': 621, '9': 553}


## 8. From a `Doc` to what gets embedded

The pipeline turns each `Doc` into bytes and calls the reference producer's `prepare_card`:

* **dataset docs** (`text`): the "object" is the UTF-8 encoding of the text, MIME `text/plain`. `object_sha256` is the
  hash of those bytes, the same value a `.txt` file with that text would give.
* **folder docs** (`path`): the file's own bytes; the MIME type is detected and text is extracted (pdftotext, pandoc, ...).

Title priority: a dataset title (BEIR) or metadata title, otherwise a synthetic `"<source> <doc_id>"` (or the filename for
folders). The embedding preimage is `title+body/v2`: the title is prepended **only** when it is a real title
(`title_source` not `synthetic`/`filename`). No model is needed for this step.

In [15]:
import argparse
from mintkit import _ref
from mintkit.cli import MINT_DEFAULTS

parser = argparse.ArgumentParser()
_ref.add_run_options(parser)
args = parser.parse_args(["--issued-unix", "1782071790"])
for k, v in MINT_DEFAULTS.items():
    setattr(args, k, v)

def prepare(source, doc):
    return _ref.prepare_card(
        doc.text.encode("utf-8"), f"{source.name}/{doc.doc_id}", args, mime="text/plain",
        dataset_title=doc.title, synthetic_title=f"{source.name} {doc.doc_id}",
    )

for source, doc in ((mldr, docs[0]), (b, next(b.iter_docs()))):
    p = prepare(source, doc)
    print(f"{source.name} {doc.doc_id}")
    print(f"  title={p.segments.title!r} title_source={p.title_source} mime={p.mime} bytes={p.nbytes}")
    print(f"  object_sha256={p.object_sha256.hex()[:16]}... wants_vector={p.wants_vector}")
    print(f"  preimage starts: {p.preimage[:90]!r}")

mldr:en doc-en-8
  title='mldr:en doc-en-8' title_source=synthetic mime=text/plain bytes=33843
  object_sha256=6a5f201173177965... wants_vector=True
  preimage starts: 'In computer programming, is a ternary operator that is part of the syntax for basic condit'
beir:tinybeir d1
  title='Cats' title_source=dataset mime=text/plain bytes=23
  object_sha256=157d615ca36e1f57... wants_vector=True
  preimage starts: 'Cats Cats purr when content.'


## 9. Writing your own source

One subclass and one registry line. Implement `iter_docs` as a generator, and `describe` so the manifest can say what
was read (include file hashes or a dataset version so runs can be reproduced). Add `splits`/`queries`/`qrels` if you have
judgments. Below, an in-memory source is registered under `mem:` and sampled like any other source.

In [16]:
class MemorySource(Source):
    def __init__(self, value, *, rows=None):
        self.name = f"mem:{value}"
        self.rows = rows or {}
    def iter_docs(self):
        for doc_id, text in self.rows.items():
            yield Doc(doc_id, text=text)
    def splits(self):
        return ["default"]
    def queries(self, split):
        return [Query("q1", "first")]
    def qrels(self, split):
        return {"q1": {"x1": 1}}
    def describe(self):
        return {"rows": len(self.rows)}

SOURCES["mem"] = MemorySource
mem = load_source("mem:demo", rows={f"x{i}": f"document {i}" for i in range(1, 8)})
sel = select(mem, "qrels:1+2", split="default")
print([d.doc_id for d in sel.iter_docs()], sel.qrels)
del SOURCES["mem"]   # leave the registry as we found it

['x1', 'x6', 'x7'] {'q1': {'x1': 1}}


## Where this goes next

```sh
python -m mintkit sources                 # list the source kinds
python -m mintkit sources mldr:en         # name and splits of one source
python -m mintkit mint mldr:en --mode positives --split dev --sample qrels:10+90 --models bge --keep-chunk-vectors
```

The run directory then holds `queries.jsonl` and `qrels.tsv` for exactly the minted documents, plus `docs.tsv`
(doc_id and object hash), so `mintkit eval` needs nothing but the run directory.

In [17]:
import shutil
shutil.rmtree(TMP)
print("removed", TMP)

removed /tmp/mintkit-sources-55kcak0_
